# Data Cleaning

This notebook keeps the original dataset unchanged and creates a cleaned copy for analysis and modeling.

In [ ]:
import pandas as pd
from pathlib import Path

input_path = Path('../data/lung_cancer_dataset.csv')
output_path = Path('../data/lung_cancer_dataset_cleaned.csv')
df = pd.read_csv(input_path)
print(f'Original shape: {df.shape}')
display(df.head())

## Alterations made

1. Remove leading/trailing spaces from column names and text values.
2. Convert `Diagnosis_Date` from text to a datetime value.
3. Convert every Yes/No field to numeric 1/0, including the target `Survived`.
4. Convert numeric columns explicitly to numeric values.
5. Check missing values, duplicate rows, duplicate patient IDs, and invalid dates.
6. Keep ordinary categories such as gender, region, treatment, and cancer stage as text because they have more than two meaningful categories.
7. Save the result as a new cleaned CSV; the raw CSV is not overwritten.

In [ ]:
# 1. Clean column names and text values
df.columns = df.columns.str.strip()
text_columns = df.select_dtypes(include='object').columns
for column in text_columns:
    df[column] = df[column].str.strip()

# 2. Convert the date column
df['Diagnosis_Date'] = pd.to_datetime(df['Diagnosis_Date'], errors='coerce')

# 3. Convert all binary Yes/No columns to 1/0
binary_columns = [
    'Secondhand_Smoke', 'Family_History', 'Occupational_Hazard',
    'Chronic_Lung_Disease', 'Asbestos_Exposure', 'Radon_Exposure',
    'Previous_Cancer_History', 'Coughing', 'Shortness_of_Breath',
    'Chest_Pain', 'Coughing_Blood', 'Fatigue', 'Weight_Loss',
    'Wheezing', 'Recurrent_Infections', 'Swallowing_Difficulty',
    'Finger_Clubbing', 'Metastasis', 'Survived'
]
for column in binary_columns:
    df[column] = df[column].map({'Yes': 1, 'No': 0})

# 4. Make sure quantitative columns are numeric
numeric_columns = [
    'Diagnosis_Year', 'Age', 'Cigarettes_Per_Day', 'Years_Smoking',
    'Pack_Years', 'BMI', 'Risk_Factor_Count', 'Symptom_Count',
    'Tumor_Size_cm', 'Survival_Months'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Binary columns converted to 1/0:')
print(binary_columns)
display(df.head())

## Validation checks

No missing values need to be filled in this dataset, but the checks are kept so the cleaning process remains safe if the source file changes later.

In [ ]:
checks = {
    'Missing values': int(df.isna().sum().sum()),
    'Duplicate rows': int(df.duplicated().sum()),
    'Duplicate Patient_ID values': int(df['Patient_ID'].duplicated().sum()),
    'Invalid dates': int(df['Diagnosis_Date'].isna().sum()),
    'Invalid ages': int(((df['Age'] < 0) | (df['Age'] > 120)).sum()),
    'Invalid BMI values': int(((df['BMI'] <= 0) | (df['BMI'] > 80)).sum()),
    'Invalid tumor sizes': int((df['Tumor_Size_cm'] <= 0).sum()),
    'Invalid survival months': int((df['Survival_Months'] <= 0).sum())
}
display(pd.Series(checks, name='Count').to_frame())

# Confirm that binary columns contain only 0 and 1 after cleaning.
for column in binary_columns:
    print(f'{column}: {sorted(df[column].dropna().unique().tolist())}')

## Convert selected categorical columns to understandable numeric values

In [ ]:
# The mappings use 0 for the lowest/no level and larger numbers for higher levels.
# Alcohol_Use: 0 = No Alcohol, 1 = Moderate, 2 = Heavy.
# Air_Pollution_Exposure: 0 = Low, 1 = Moderate, 2 = High.
# Exercise_Frequency: 0 = Low, 1 = Moderate, 2 = High.
# Smoking_Status: 0 = Never Smoked, 1 = Former Smoker, 2 = Current Smoker.

category_mappings = {
    'Alcohol_Use': {'No Alcohol': 0, 'Moderate': 1, 'Heavy': 2},
    'Air_Pollution_Exposure': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Exercise_Frequency': {'Low': 0, 'Moderate': 1, 'High': 2},
    'Smoking_Status': {'Never Smoked': 0, 'Former Smoker': 1, 'Current Smoker': 2}
}

for column, mapping in category_mappings.items():
    df[column] = df[column].map(mapping)

display(df[list(category_mappings)].head())
print('Category mappings applied successfully.')

# Remove columns that will not be used for feature engineering or ML.
columns_to_exclude = [
    'Survival_Months',
    'Metastasis',
    'Tumor_Size_cm',
    'Patient_ID',
    'Age_Group',
    'BMI_Category',
    'Pack_Years',
    'Risk_Factor_Count',
    'Symptom_Count',
    'Diagnosis_Method',
    'Treatment_Chemo + Radiation',
    'Air_Pollution_Exposure', 
    'Alcohol_Use', 'Exercise_Frequency', 
    'Treatment_Surgery + Chemotherapy', 
    'Smoking_Status', 'Fatigue'
]

# Add the original feature names printed by Model_building.ipynb when
# their tree importance is below 0.02. Keep this list editable.
low_importance_columns = [
    # Example: 'Some_Feature',
]
columns_to_exclude.extend(low_importance_columns)
columns_to_exclude = list(dict.fromkeys(columns_to_exclude))

df_cleaned = df.drop(columns=columns_to_exclude, errors='ignore')
df_cleaned.to_csv('../data/lung_cancer_dataset_cleaned.csv', index=False)

print('Excluded columns:')
print(columns_to_exclude)
print(f'Cleaned file saved to: ../data/lung_cancer_dataset_cleaned.csv')
print(f'Final shape: {df_cleaned.shape}')
display(df_cleaned.head())